# CT Scanner Simulator

**Group:**
* Jakub Biernat 160248
* Eryk Masian 160228

**Tomograph Model:**
* **Cone-beam**

**Technologies Used:**
* **Language:** Python
* **Libraries:** numpy, matplotlib, ipywidgets, pydicom

**Functions**
* [Radon Transform](../src/radon_transform.py)
* [Emitter Positioning](../src/emiters.py)
* [Detector Positioning](../src/detectors.py)
* [Bresenham Algorithm]()

## Imports

In [17]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import io

# ---------------- OBRAZY ----------------
available_images = {
    "CT_ScoutView": "../data/input/CT_ScoutView.jpg",
    "CT_ScoutView-large": "../data/input/CT_ScoutView-large.jpg",
    "Kolo": "../data/input/Kolo.jpg",
    "Kropka": "../data/input/Kropka.jpg",
    "Kwadraty2": "../data/input/Kwadraty2.jpg",
    "Paski2": "../data/input/Paski2.jpg",
    "SADDLE_PE": "../data/input/SADDLE_PE.jpg",
    "SADDLE_PE-large": "../data/input/SADDLE_PE-large.jpg",
    "Shepp_logan": "../data/input/Shepp_logan.jpg",
    "CT_ScoutView_DCM": "../data/input/CT_ScoutView.dcm",
    "CT_ScoutView-large_DCM": "../data/input/CT_ScoutView-large.dcm",
    "Kolo_DCM": "../data/input/Kolo.dcm",
    "Kropka_DCM": "../data/input/Kropka.dcm",
    "Kwadraty2_DCM": "../data/input/Kwadraty2.dcm",
    "Paski2_DCM": "../data/input/Paski2.dcm",
    "SADDLE_PE_DCM": "../data/input/SADDLE_PE.dcm",
    "SADDLE_PE-large_DCM": "../data/input/SADDLE_PE-large.dcm",
    "Shepp_logan_DCM": "../data/input/Shepp_logan.dcm"
}

# ---------------- STAŁE ----------------
FIGSIZE = (6, 6)
DPI = 150
PREVIEW_EMITTER_POS = 90  # kąt emitera do pokazywania

# ---------------- UI ----------------
image_selector = widgets.Dropdown(
    options=list(available_images.keys()),
    value=list(available_images.keys())[0],
    description="Obraz:"
)

detectors_slider = widgets.IntSlider(
    value=16, min=2, max=128, step=1,
    description="Detektory:"
)

fan_slider = widgets.FloatSlider(
    value=30, min=1, max=270, step=1,
    description="Wachlarz:"
)

scans_slider = widgets.IntSlider(
    value=360,
    min=90,
    max=720,
    step=90,
    description="Liczba skanów:"
)

alpha_label = widgets.Label()
canvas = widgets.Image(format="png")
info_out = widgets.Output()

# 🔥 STAŁY ROZMIAR WIDŻETU
canvas.layout = widgets.Layout(
    width="300px",
    height="300px"
)

left_panel = widgets.VBox([
    image_selector,
    detectors_slider,
    fan_slider,
    widgets.HBox([scans_slider, alpha_label])
], layout=widgets.Layout(width="25%"))

middle_panel = widgets.VBox([
    canvas
], layout=widgets.Layout(
    width="40%",
    align_items="center",
    justify_content="center"
))

right_panel = widgets.VBox([
    info_out
], layout=widgets.Layout(width="35%"))

ui = widgets.HBox([
    left_panel,
    middle_panel,
    right_panel
])

display(ui)

# ---------------- HELPERS ----------------
def fig_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=DPI)
    buf.seek(0)
    return buf.read()

# ---------------- UPDATE ----------------
def update(*args):
    name = image_selector.value
    path = available_images[name]

    info, image = load_image(path)

    detectors = detectors_slider.value
    fan = fan_slider.value
    angle = PREVIEW_EMITTER_POS  # 🔥 stały kąt

    xe, ye = emitter_pos(image, angle)
    dets = detectors_pos(image, angle, detectors, fan)

    scans = scans_slider.value
    angle = 360 / scans  # α

    alpha_label.value = f"α = {angle:.3f}°"

    # ---------------- RYSOWANIE ----------------
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

    ax.imshow(image, cmap="gray", aspect="equal")

    # emitter
    ax.scatter(xe, ye, c="red", s=80)

    # detektory + linie
    if len(dets) > 0:
        dx, dy = zip(*dets)
        ax.scatter(dx, dy, c="blue", s=30)

        for x, y in dets:
            ax.plot([xe, x], [ye, y],
                    c="red", linewidth=0.5, alpha=0.6)

    ax.set_title(name)
    ax.axis("off")

    # ---------------- UPDATE WIDŻETA ----------------
    canvas.value = fig_to_png(fig)
    plt.close(fig)

    # ---------------- INFO ----------------
    with info_out:
        info_out.clear_output(wait=True)
        print("=== INFO DICOM / IMAGE ===")
        for k, v in info.items():
            print(f"{k}: {v}")

# ---------------- OBSERWATORY ----------------
for w in [image_selector, detectors_slider, fan_slider, scans_slider]:
    w.observe(update, "value")

# start
update()

In [11]:
import os
import sys
import io
import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image as PILImage

import pydicom
from pydicom.dataset import FileDataset, FileMetaDataset
from pydicom.uid import UID

sys.path.append(os.path.abspath('..'))

from src.radon_transform import get_sinogram, inverse_radon
from src.bresenham import bresenham
from src.emiters import emitter_pos
from src.detectors import detectors_pos
from src.filter import apply_filter
from src.rmse import calculate_rmse
from src.handle_dicoms import load_image
from src.detectors import detectors_pos

In [62]:
PLOT_PX = 500
DPI = 100
FIG_IN = PLOT_PX / DPI

available_images = {
    "CT_ScoutView": "../data/input/CT_ScoutView.jpg",
    "CT_ScoutView-large": "../data/input/CT_ScoutView-large.jpg",
    "Kolo": "../data/input/Kolo.jpg",
    "Kropka": "../data/input/Kropka.jpg",
    "Kwadraty2": "../data/input/Kwadraty2.jpg",
    "Paski2": "../data/input/Paski2.jpg",
    "SADDLE_PE": "../data/input/SADDLE_PE.jpg",
    "SADDLE_PE-large": "../data/input/SADDLE_PE-large.jpg",
    "Shepp_logan": "../data/input/Shepp_logan.jpg",
    "CT_ScoutView_DCM": "../data/input/CT_ScoutView.dcm",
    "CT_ScoutView-large_DCM": "../data/input/CT_ScoutView-large.dcm",
    "Kolo_DCM": "../data/input/Kolo.dcm",
    "Kropka_DCM": "../data/input/Kropka.dcm",
    "Kwadraty2_DCM": "../data/input/Kwadraty2.dcm",
    "Paski2_DCM": "../data/input/Paski2.dcm",
    "SADDLE_PE_DCM": "../data/input/SADDLE_PE.dcm",
    "SADDLE_PE-large_DCM": "../data/input/SADDLE_PE-large.dcm",
    "Shepp_logan_DCM": "../data/input/Shepp_logan.dcm"
}

current_image_name = None
latest_reconstructed_image = None
full_sinogram = None
reconstruction_history = None
_bg_pil = None
image_path = None
image_info = None
image_array = None
image_name = None

In [64]:
image_dropdown = widgets.Dropdown(options=list(available_images.keys()), value="Shepp_logan", description='Obraz:')
preview_image_output = widgets.Output()
preview_info_output = widgets.Output()

def update_image_preview(change):
    """Updates image preview and loads image into image_array and image_info"""
    global image_name, image_array, image_path, image_info

    image_name = change.new
    image_path = available_images[image_name]

    try:
        image_info, image_array = load_image(image_path)

        with preview_image_output:
            clear_output(wait=True)

            plt.figure(figsize=(4, 4))
            plt.imshow(image_array, cmap='gray', aspect='equal')
            plt.title(f"Wybrano: {image_name}")
            plt.axis('off')
            plt.show()

        with preview_info_output:
            clear_output(wait=True)

            for info_id, info_value in image_info.items():
                print(f"{info_id}: {info_value}")

    except FileNotFoundError:
        clear_output(wait=True)
        print(f"Błąd: Nie znaleziono pliku {image_path}")

image_dropdown.observe(update_image_preview, names='value')
ui = widgets.HBox([
    preview_image_output,
    preview_info_output
])
display(image_dropdown, ui)
update_image_preview(type('obj', (object,), {'new': image_dropdown.value}))

Dropdown(description='Obraz:', index=8, options=('CT_ScoutView', 'CT_ScoutView-large', 'Kolo', 'Kropka', 'Kwad…

(1024, 1024)


In [67]:
90 = 90
detectors_slider = widgets.IntSlider(min=10, max=720, step=10, value=180, description='Detektory:')
span_slider      = widgets.IntSlider(min=45, max=270, step=45, value=180, description='Rozpiętość:')
title_label      = widgets.Label(value='')
canvas           = widgets.Image(format='png', layout=widgets.Layout(width=f'{PLOT_PX}px', height=f'{PLOT_PX}px'))


def _fig_to_pil(fig, transparent=False) -> PILImage:
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=DPI, transparent=transparent, bbox_inches=None)
    plt.close(fig)
    buf.seek(0)
    return PILImage.open(buf).convert('RGBA')


def _pil_to_bytes(img: PILImage) -> bytes:
    buf = io.BytesIO()
    img.save(buf, format='png')
    buf.seek(0)
    return buf.read()


def render_background():
    global _bg_pil

    if image_array is None:
        return

    H, W = image_array.shape

    fig = plt.figure(figsize=(FIG_IN, FIG_IN), dpi=DPI)
    ax  = fig.add_axes([0, 0, 1, 1])

    ax.imshow(image_array, cmap='gray', extent=[0, W, H, 0], aspect='equal')

    ax.legend(
        handles=[
            Line2D([0],[0], marker='o', color='w', markerfacecolor='r', markersize=8, label='Emiter'),
            Line2D([0],[0], marker='o', color='w', markerfacecolor='b', markersize=4, label='Detektory'),
        ],
        loc='upper right', framealpha=0.6, facecolor='#111',
        labelcolor='white', fontsize=9
    )

    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.axis('off')

    _bg_pil = _fig_to_pil(fig, transparent=False)

def update_geometry(change=None):
    global image_array, _bg_pil

    if image_array is None or _bg_pil is None:
        return

    H, W = image_array.shape

    detectors = detectors_slider.value
    span      = span_slider.value

    xe, ye = emitter_pos(image_array, 90)
    det_pos = detectors_pos(image_array, 90, detectors, span)

    fig = plt.figure(figsize=(FIG_IN, FIG_IN), dpi=DPI)
    ax  = fig.add_axes([0, 0, 1, 1])

    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)

    if det_pos:
        xd, yd = zip(*det_pos)

        ax.add_collection(LineCollection(
            [[(xe, ye), (xi, yi)] for xi, yi in det_pos],
            colors='r', alpha=0.15, linewidths=0.8
        ))

        ax.plot(xd, yd, 'bo', markersize=4, markeredgewidth=0)

    ax.plot([xe], [ye], 'ro', markersize=8)

    margin = 0.3
    ax.set_xlim(-W * margin, W * (1 + margin))
    ax.set_ylim(H * (1 + margin), -H * margin)
    ax.axis('off')

    overlay_pil = _fig_to_pil(fig, transparent=True)

    composite = _bg_pil.copy()
    composite.paste(overlay_pil, (0, 0), mask=overlay_pil)

    title_label.value = (
        f'Kąt: {90}°  |  detektory: {detectors}  |  rozpiętość: {span}°'
    )

    canvas.value = _pil_to_bytes(composite)

detectors_slider.observe(update_geometry, names='value')
span_slider.observe(update_geometry, names='value')

display(widgets.VBox([detectors_slider, span_slider, title_label, canvas]))

if image_array is not None:
    render_background()
    update_geometry()

In [5]:
scans_input = widgets.IntSlider(min=90, max=720, step=90, value=180, description='L. skanów:')
filter_checkbox = widgets.Checkbox(value=False, description='Użyj filtru splotowego')
run_button = widgets.Button(description="Uruchom Skanowanie", button_style='info')

history_slider = widgets.IntSlider(min=1, max=180, step=1, value=180, description='Iteracja:', layout=widgets.Layout(width='600px'))
player_ui = widgets.VBox([history_slider])
player_ui.layout.display = 'none'

results_output = widgets.Output()
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='Postęp:', bar_style='info')

def execute_scan(b):
    global full_sinogram, reconstruction_history, latest_reconstructed_image

    with results_output:
        clear_output(wait=True)
        if get_image_array() is None:
            print("Wybierz obraz w drugiej komórce!")
            return

        detectors = detectors_slider.value
        span = span_slider.value
        scans = scans_input.value
        use_filter = filter_checkbox.value

        player_ui.layout.display = 'none'
        progress_bar.value = 10
        display(progress_bar)

        progress_bar.description = 'Sinogram...'
        full_sinogram = get_sinogram(get_image_array(), detectors, scans, span)
        progress_bar.value = 40

        progress_bar.description = 'Filtrowanie...'
        if use_filter:
            full_sinogram = apply_filter(full_sinogram)
        progress_bar.value = 50

        progress_bar.description = 'Rekonstrukcja...'
        reconstruction_history = inverse_radon(full_sinogram, get_image_array().shape, detectors, scans, span, return_history=True)
        progress_bar.value = 90

        latest_reconstructed_image = reconstruction_history[-1]

        progress_bar.value = 100
        progress_bar.description = 'Gotowe!'
        history_slider.max = scans
        history_slider.value = scans
        player_ui.layout.display = 'block'

        setup_player()

def setup_player():
    clear_output(wait=True)
    image_widget = widgets.Image(format='png', width=900, height=350)

    plt.ioff()
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(get_image_array(), cmap='gray')
    axes[0].set_title("Obraz wejściowy")
    axes[0].axis('off')

    im_sino = axes[1].imshow(np.zeros_like(full_sinogram), cmap='gray', aspect='auto', vmin=0, vmax=np.max(full_sinogram))
    axes[1].set_title("Sinogram (Tworzenie)")
    axes[1].axis('off')

    im_recon = axes[2].imshow(np.zeros_like(get_image_array()), cmap='gray', vmin=0, vmax=255)
    axes[2].set_title("Rekonstrukcja")
    axes[2].axis('off')
    plt.tight_layout()

    def update_frame(change):
        step = history_slider.value

        partial_sinogram = np.zeros_like(full_sinogram)
        partial_sinogram[:step, :] = full_sinogram[:step, :]

        im_sino.set_data(partial_sinogram)
        im_recon.set_data(reconstruction_history[step - 1])

        current_rmse = calculate_rmse(get_image_array(), reconstruction_history[step - 1])
        fig.suptitle(f"Krok: {step}/{history_slider.max} | Ostatnie RMSE: {current_rmse:.4f}", fontsize=14)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        buf.seek(0)
        image_widget.value = buf.read()
        buf.close()

    history_slider.observe(update_frame, names='value')
    display(widgets.VBox([player_ui, image_widget]))
    update_frame(None)

run_button.on_click(execute_scan)
display(widgets.VBox([scans_input, filter_checkbox, run_button, results_output]))